# 06 - Eval

Ragas generation-quality evaluation (faithfulness, answer relevance) for the RAG
pipeline's written answers.

The retrieval-method comparison this notebook originally did (dense vs BM25 vs
hybrid vs reranker candidates, MRR/hit@k) has moved to
`04b_retrieval_sealion_rerank.ipynb`, which reran the same labeled queries with
SEA-LION added and settled the decision (SEA-LION reranker, MRR 0.950 — see that
notebook's comparison cell). This notebook now covers only what that one
doesn't: whether the *generated answer* is faithful to and relevant to the
retrieved context, using Ragas.

## Step 1: Setup

Imports, reconnect to the Chroma collection, load `chunks.json` /
`embeddings_bge_m3.npy` / `chunk_ids_bge_m3.json`, and rebuild the retrieval
functions (dense / BM25 / hybrid / hybrid+rerank) — reranker is SEA-LION, the
method chosen in `04b_retrieval_sealion_rerank.ipynb`, not a comparison of
candidates.

In [1]:
import json
import re
from pathlib import Path

import chromadb
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

CHUNKS_PATH = Path("../data/processed/chunks.json")
EMBEDDINGS_PATH = Path("../data/processed/embeddings_bge_m3.npy")
CHUNK_IDS_PATH = Path("../data/processed/chunk_ids_bge_m3.json")
CHROMA_DIR = Path("../data/processed/chroma")

EMBEDDING_MODEL_NAME = "BAAI/bge-m3"
# Final reranker choice -- 04b_retrieval_sealion_rerank.ipynb's benchmark showed
# SEA-LION (MRR 0.950) beats both ms-marco-MiniLM-L-6-v2 (MRR 0.425) and
# bge-reranker-v2-m3 (MRR 0.817), at effectively the same latency cost as the
# latter. It's a bi-encoder (cosine similarity), not a cross-encoder like those
# two -- see hybrid_rerank_retrieve below for how reranking differs as a result.
RERANKER_MODEL_NAME = "aisingapore/SEA-LION-E5-Embedding-600M"

chunks = json.loads(CHUNKS_PATH.read_text(encoding="utf-8"))
chunks_by_id = {chunk["chunk_id"]: chunk for chunk in chunks}

embeddings = np.load(EMBEDDINGS_PATH)
chunk_ids = json.loads(CHUNK_IDS_PATH.read_text(encoding="utf-8"))
chunk_id_to_idx = {cid: i for i, cid in enumerate(chunk_ids)}

client = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection = client.get_collection(name="bge_m3")

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
reranker_model = SentenceTransformer(RERANKER_MODEL_NAME)


def tokenize(text: str) -> list[str]:
    return re.findall(r"\w+", text.lower())


bm25 = BM25Okapi([tokenize(chunks_by_id[cid]["text"]) for cid in chunk_ids])


def dense_retrieve(query: str, top_k: int = 5) -> list[str]:
    query_embedding = embedding_model.encode(query, convert_to_numpy=True)
    results = collection.query(query_embeddings=[query_embedding.tolist()], n_results=top_k)
    return list(results["ids"][0])


def bm25_retrieve(query: str, top_k: int = 5) -> list[str]:
    scores = bm25.get_scores(tokenize(query))
    ranked = sorted(zip(chunk_ids, scores), key=lambda x: x[1], reverse=True)[:top_k]
    return [cid for cid, _ in ranked]


def hybrid_retrieve(query: str, top_k: int = 5) -> list[str]:
    dense_ids = dense_retrieve(query, top_k=10)
    bm25_ids = bm25_retrieve(query, top_k=10)
    scores: dict[str, float] = {}
    for ranked in (dense_ids, bm25_ids):
        for rank, cid in enumerate(ranked, start=1):
            scores[cid] = scores.get(cid, 0.0) + 1.0 / (60 + rank)
    return [cid for cid, _ in sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k]]


def hybrid_rerank_retrieve(query: str, top_k: int = 5) -> list[str]:
    candidates = hybrid_retrieve(query, top_k=10)
    # STS is the only named prompt documented on the SEA-LION-E5 model card,
    # used for both query and passage since this only needs a symmetric
    # cosine-similarity score for reranking, not asymmetric retrieval.
    query_embedding = reranker_model.encode(query, convert_to_numpy=True, prompt_name="STS")
    candidate_embeddings = reranker_model.encode(
        [chunks_by_id[cid]["text"] for cid in candidates],
        convert_to_numpy=True,
        prompt_name="STS",
    )
    similarities = candidate_embeddings @ query_embedding / (
        np.linalg.norm(candidate_embeddings, axis=1) * np.linalg.norm(query_embedding)
    )
    reranked = sorted(zip(candidates, similarities), key=lambda x: x[1], reverse=True)
    return [cid for cid, _ in reranked[:top_k]]


RETRIEVAL_METHODS = {
    "dense": dense_retrieve,
    "bm25": bm25_retrieve,
    "hybrid": hybrid_retrieve,
    "hybrid_rerank": hybrid_rerank_retrieve,
}

len(chunks), collection.count(), list(RETRIEVAL_METHODS)

c:\Users\rames\Documents\GitHub\migrantBuddy\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 4874.68it/s]


(16, 16, ['dense', 'bm25', 'hybrid', 'hybrid_rerank'])

## Step 2: Labeled eval set

`(query, correct_chunk_id)` pairs for the sample queries — writing down the
answers we already know from reading the docs, instead of eyeballing results.

In [2]:
def find_chunk_id(document_slug: str, heading_contains: str) -> str:
    matches = [
        cid
        for cid in chunk_ids
        if document_slug in chunks_by_id[cid]["url"]
        and heading_contains.lower() in chunks_by_id[cid]["heading_path"].lower()
    ]
    assert len(matches) == 1, f"Expected exactly 1 match for {heading_contains!r}, got {matches}"
    return matches[0]


def find_chunk_id_by_document(document_slug: str) -> str:
    # For pages with no heading structure (e.g. contact-us) -- find_chunk_id's
    # heading lookup doesn't apply, so just match the (single) chunk for that doc.
    matches = [cid for cid in chunk_ids if document_slug in chunks_by_id[cid]["url"]]
    assert len(matches) == 1, f"Expected exactly 1 chunk for {document_slug!r}, got {matches}"
    return matches[0]


# Same rough, unverified queries as 04_retrieval/05_generation, now with a labeled
# correct answer per query -- looked up by heading text, not guessed by index.
#
# Burmese and Thai dropped from this eval set (not from the corpus/retrieval
# benchmarks elsewhere) -- both showed low Ragas scores traceable to judge/
# generation noise rather than retrieval quality: Thai's faithfulness=0.0
# repeats identically regardless of retrieval method (ms-marco, bge, SEA-LION),
# pointing at the llama3.1:8b judge failing to verify Thai-language claims
# against English context, not a real faithfulness problem. Burmese swung from
# faithfulness=1.0 to 0.25 between runs with no code change other than
# generation happening again (no temperature/seed pinned on the Ollama call),
# consistent with generation randomness rather than a retrieval regression.
# Keeping them in a 2-query eval set would let judge/generation noise dominate
# the average rather than the actual pipeline quality signal this step exists
# to measure.
#
# Language coverage: same two anchor questions (overtime pay, salary timing)
# translated into more languages, so language is the only variable that changes.
# Category coverage: one new English question per newly-added corpus category
# (work-permit, medical, help), so topic and language aren't tested at once.
LABELED_QUERIES = [
    {
        "language": "en",
        "text": "How much overtime pay am I entitled to?",
        "correct_chunk_id": find_chunk_id("hours-of-work", "Overtime pay"),
    },
    {
        "language": "en",
        "text": "When must my employer pay my salary?",
        "correct_chunk_id": find_chunk_id("paying-salary", "How often salary must be paid"),
    },
    {
        "language": "ms",
        "text": "Bilakah majikan saya perlu bayar gaji saya?",
        "correct_chunk_id": find_chunk_id("paying-salary", "How often salary must be paid"),
    },
    {
        "language": "ta",
        "text": "எனக்கு எவ்வளவு கூடுதல் நேர ஊதியம் கிடைக்கும்?",
        "correct_chunk_id": find_chunk_id("hours-of-work", "Overtime pay"),
    },
    {
        "language": "vi",
        "text": "Chủ sử dụng lao động của tôi phải trả lương khi nào?",
        "correct_chunk_id": find_chunk_id("paying-salary", "How often salary must be paid"),
    },
    {
        "language": "en",
        "text": "Who pays repatriation costs when my Work Permit ends?",
        "correct_chunk_id": find_chunk_id("work-permit-conditions", "employment ends"),
    },
    {
        "language": "en",
        "text": "How much medical insurance must my employer provide?",
        "correct_chunk_id": find_chunk_id("medical-insurance", "should cover"),
    },
    {
        "language": "en",
        "text": "How can I contact MOM?",
        "correct_chunk_id": find_chunk_id_by_document("contact-us"),
    },
]

for q in LABELED_QUERIES:
    print(f"[{q['language']}] {q['text']}\n  -> {q['correct_chunk_id']}\n")

[en] How much overtime pay am I entitled to?
  -> https-www-mom-gov-sg-employment-practices-hours-of-work-overtime-and-rest-days::chunk-2

[en] When must my employer pay my salary?
  -> https-www-mom-gov-sg-employment-practices-salary-paying-salary::chunk-1

[ms] Bilakah majikan saya perlu bayar gaji saya?
  -> https-www-mom-gov-sg-employment-practices-salary-paying-salary::chunk-1

[ta] எனக்கு எவ்வளவு கூடுதல் நேர ஊதியம் கிடைக்கும்?
  -> https-www-mom-gov-sg-employment-practices-hours-of-work-overtime-and-rest-days::chunk-2

[vi] Chủ sử dụng lao động của tôi phải trả lương khi nào?
  -> https-www-mom-gov-sg-employment-practices-salary-paying-salary::chunk-1

[en] Who pays repatriation costs when my Work Permit ends?
  -> https-www-mom-gov-sg-passes-and-permits-work-permit-for-foreign-worker-sector-specific-rules-work-permit-conditions::chunk-3

[en] How much medical insurance must my employer provide?
  -> https-www-mom-gov-sg-passes-and-permits-work-permit-for-foreign-worker-sector-sp

## Step 3: Ragas setup

Install (see `requirements.txt`) and configure Ragas to use local models only —
`qwen3:8b` via Ollama as the judge LLM, BGE-M3 as the embedding model — instead of
Ragas's OpenAI default, per CLAUDE.md's local-first dev decision.

In [3]:
OLLAMA_MODEL_NAME = "qwen3:8b"
JUDGE_MODEL_NAME = "llama3.1:8b"  # separate from the generation model -- no thinking-mode wrapper
OLLAMA_BASE_URL = "http://localhost:11434"

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.llms import LangchainLLMWrapper

judge_llm = LangchainLLMWrapper(ChatOllama(model=JUDGE_MODEL_NAME, base_url=OLLAMA_BASE_URL))
ragas_embeddings = LangchainEmbeddingsWrapper(HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME))

judge_llm, ragas_embeddings

C:\Users\rames\AppData\Local\Temp\ipykernel_15468\4016536569.py:10: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  judge_llm = LangchainLLMWrapper(ChatOllama(model=JUDGE_MODEL_NAME, base_url=OLLAMA_BASE_URL))
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 121201.16it/s]
C:\Users\rames\AppData\Local\Temp\ipykernel_15468\4016536569.py:11: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME))


(LangchainLLMWrapper(langchain_llm=ChatOllama(...)),
 LangchainEmbeddingsWrapper(embeddings=HuggingFaceEmbeddings(...)))

## Step 4: Build the eval dataset

For each labeled query, actually run retrieve → generate (reusing `05_generation`'s
logic) to get real `{question, contexts, answer}` triples, packaged into the shape
Ragas expects.

In [4]:
import requests

SYSTEM_PROMPT = """You are migrantBuddy, an assistant that answers questions about \
Singapore employment rules (work passes, salary, working hours) for migrant workers.

Answer ONLY using the information in the provided context. If the context does not \
contain enough information to answer the question, say so clearly instead of \
guessing. Do not use any outside knowledge. Keep answers clear and concise, \
suitable for someone who may not be a native English speaker."""


def build_prompt(query: str, context_chunks: list[dict]) -> str:
    context_text = "\n\n---\n\n".join(
        f"Source: {chunk['url']}\n{chunk['text']}" for chunk in context_chunks
    )
    return f"""Context:
{context_text}

Question: {query}

Answer:"""


def generate_answer(query: str, method: str = "hybrid_rerank", top_k: int = 5) -> dict:
    retrieved_ids = RETRIEVAL_METHODS[method](query, top_k)
    context_chunks = [chunks_by_id[cid] for cid in retrieved_ids]
    user_prompt = build_prompt(query, context_chunks)

    response = requests.post(
        f"{OLLAMA_BASE_URL}/api/chat",
        json={
            "model": OLLAMA_MODEL_NAME,
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt},
            ],
            "stream": False,
        },
        timeout=120,
    )
    response.raise_for_status()
    answer = response.json()["message"]["content"]

    return {
        "question": query,
        "contexts": [chunk["text"] for chunk in context_chunks],
        "answer": answer,
    }


eval_dataset_rows = [generate_answer(q["text"], method="hybrid_rerank") for q in LABELED_QUERIES]
eval_dataset_rows[0]

{'question': 'How much overtime pay am I entitled to?',
 'contexts': ['Hours of work, overtime and rest day > Overtime pay\n\nOvertime work is all work in excess of the normal hours of work (excluding breaks).\n\nYou can claim overtime if you are:\n\n- A non-workman earning a monthly basic salary of $2,600 or less.\n- A workman earning a monthly basic salary of $4,500 or less. \n\nThe overtime rate payable for non-workmen is **capped at the salary level of $2,600, or an hourly rate of $13.60**.\n\nFor overtime work, your employer must pay you **at least 1.5 times** the hourly basic rate of pay. Payment must be made **within 14 days** after the last day of the salary period.\n\nA non-workman earns $2,600 a month and works 2 hours of overtime. The overtime pay is:\n\n$13.60 × 1.5 × 2 hours = $40.80\n\nCalculate your overtime pay\n\nOvertime pay is calculated as follows:\n\n- Hourly basic rate of pay × 1.5 × number of hours worked overtime\n\nThe hourly basic rate of pay is calculated as 

## Step 5: Run Ragas metrics

`Faithfulness` and `AnswerRelevancy` specifically — matching CLAUDE.md's stated
generation-eval scope. Retrieval quality (MRR/hit@k) is covered separately in
`04b_retrieval_sealion_rerank.ipynb`, not repeated here.

In [5]:
import asyncio

from ragas import SingleTurnSample
from ragas.metrics import AnswerRelevancy, Faithfulness

samples = [
    SingleTurnSample(
        user_input=row["question"],
        retrieved_contexts=row["contexts"],
        response=row["answer"],
    )
    for row in eval_dataset_rows
]

faithfulness_metric = Faithfulness(llm=judge_llm)
answer_relevancy_metric = AnswerRelevancy(llm=judge_llm, embeddings=ragas_embeddings)

ragas_scores = []
for i, sample in enumerate(samples):
    query_info = LABELED_QUERIES[i]
    row = {
        "language": query_info["language"],
        "query": query_info["text"],
        "answer": eval_dataset_rows[i]["answer"],
    }

    try:
        row["faithfulness"] = await asyncio.wait_for(
            faithfulness_metric.single_turn_ascore(sample), timeout=300
        )
    except Exception as e:
        row["faithfulness"] = None
        print(f"FAILED faithfulness [{query_info['language']}] {type(e).__name__}: {e}")

    try:
        row["answer_relevancy"] = await asyncio.wait_for(
            answer_relevancy_metric.single_turn_ascore(sample), timeout=300
        )
    except Exception as e:
        row["answer_relevancy"] = None
        print(f"FAILED answer_relevancy [{query_info['language']}] {type(e).__name__}: {e}")

    ragas_scores.append(row)

ragas_scores

C:\Users\rames\AppData\Local\Temp\ipykernel_15468\643046577.py:4: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import AnswerRelevancy, Faithfulness
C:\Users\rames\AppData\Local\Temp\ipykernel_15468\643046577.py:4: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import AnswerRelevancy, Faithfulness


[{'language': 'en',
  'query': 'How much overtime pay am I entitled to?',
  'answer': 'To calculate your overtime pay in Singapore:  \n\n1. **Eligibility**:  \n   - **Non-workmen** (monthly salary ≤ $2,600) or **workmen** (monthly salary ≤ $4,500) are eligible.  \n\n2. **Overtime Rate**:  \n   - **Minimum**: 1.5 × hourly basic rate of pay.  \n   - **Cap for non-workmen**: $13.60/hour (if salary ≤ $2,600).  \n\n3. **Formula**:  \n   - Overtime pay = (Hourly basic rate × 1.5) × number of overtime hours.  \n\n4. **Examples**:  \n   - A non-workman earning $2,600/month with 2 overtime hours:  \n     $13.60 × 1.5 × 2 = **$40.80**.  \n\n5. **Payment**:  \n   - Must be paid within **14 days** of the salary period.  \n\n6. **Maximum Overtime per Month**:  \n   - **72 hours** (unless an exemption is approved).  \n\nIf you need a specific calculation, provide your salary type (monthly/daily/piece-rated), hourly rate, and overtime hours.',
  'faithfulness': 1.0,
  'answer_relevancy': np.float64(0

## Step 6: Review results

Per-query faithfulness/answer-relevancy scores, with a manual read of anything
scoring low — is it actually hallucinating, or is the metric being harsh.

In [6]:
for row in ragas_scores:
    faithfulness = row["faithfulness"]
    relevancy = row["answer_relevancy"]
    flag = ""
    if faithfulness is None or faithfulness < 0.7:
        flag = "  <-- review"
    if relevancy is None or relevancy < 0.5:
        flag = "  <-- review"
    print(
        f"[{row['language']}] faithfulness={faithfulness}  answer_relevancy={relevancy}{flag}  "
        f"{row['query'][:50]}"
    )
    # Full answer only for flagged rows -- printing all 10 in full would bury
    # the summary table; flagged rows are exactly the ones worth reading.
    if flag:
        print(f"    answer: {row['answer']}")

valid_faithfulness = [r["faithfulness"] for r in ragas_scores if r["faithfulness"] is not None]
valid_relevancy = [r["answer_relevancy"] for r in ragas_scores if r["answer_relevancy"] is not None]

print(f"\nAvg faithfulness ({len(valid_faithfulness)}/{len(ragas_scores)} scored): "
      f"{sum(valid_faithfulness) / len(valid_faithfulness):.3f}" if valid_faithfulness else "\nNo faithfulness scores succeeded")
print(f"Avg answer_relevancy ({len(valid_relevancy)}/{len(ragas_scores)} scored): "
      f"{sum(valid_relevancy) / len(valid_relevancy):.3f}" if valid_relevancy else "No answer_relevancy scores succeeded")

[en] faithfulness=1.0  answer_relevancy=0.75125869217807  How much overtime pay am I entitled to?
[en] faithfulness=1.0  answer_relevancy=0.9002961748750917  When must my employer pay my salary?
[ms] faithfulness=1.0  answer_relevancy=0.8696361287099377  Bilakah majikan saya perlu bayar gaji saya?
[ta] faithfulness=1.0  answer_relevancy=0.7397435278716212  எனக்கு எவ்வளவு கூடுதல் நேர ஊதியம் கிடைக்கும்?
[vi] faithfulness=1.0  answer_relevancy=0.7666448358657904  Chủ sử dụng lao động của tôi phải trả lương khi nà
[en] faithfulness=1.0  answer_relevancy=0.8003658331029219  Who pays repatriation costs when my Work Permit en
[en] faithfulness=1.0  answer_relevancy=0.7328844545384495  How much medical insurance must my employer provid
[en] faithfulness=0.8  answer_relevancy=0.9936476381350482  How can I contact MOM?

Avg faithfulness (8/8 scored): 0.975
Avg answer_relevancy (8/8 scored): 0.819


## Step 7: Run A, trimmed to 8 queries (dense retrieval)

Run A (`dense` retrieval) was only ever scored on the original 10-query set,
before Burmese/Thai were dropped -- unlike Run B, which has a clean 8-query
counterpart in Run C above. This reruns Run A on the same 8-query
`LABELED_QUERIES` set used by Run C and `06b_eval_sealion_generation.ipynb`,
so all three (dense / hybrid_rerank / SEA-LION generation) are comparable on
identical queries. Reuses `judge_llm`, `ragas_embeddings`,
`faithfulness_metric`, `answer_relevancy_metric` from Steps 3 and 5 --
generation model is still `qwen3:8b`, only the retrieval method changes.

In [7]:
run_a_dataset_rows = [generate_answer(q["text"], method="dense") for q in LABELED_QUERIES]

run_a_samples = [
    SingleTurnSample(
        user_input=row["question"],
        retrieved_contexts=row["contexts"],
        response=row["answer"],
    )
    for row in run_a_dataset_rows
]

run_a_scores = []
for i, sample in enumerate(run_a_samples):
    query_info = LABELED_QUERIES[i]
    row = {
        "language": query_info["language"],
        "query": query_info["text"],
        "answer": run_a_dataset_rows[i]["answer"],
    }

    try:
        row["faithfulness"] = await asyncio.wait_for(
            faithfulness_metric.single_turn_ascore(sample), timeout=300
        )
    except Exception as e:
        row["faithfulness"] = None
        print(f"FAILED faithfulness [{query_info['language']}] {type(e).__name__}: {e}")

    try:
        row["answer_relevancy"] = await asyncio.wait_for(
            answer_relevancy_metric.single_turn_ascore(sample), timeout=300
        )
    except Exception as e:
        row["answer_relevancy"] = None
        print(f"FAILED answer_relevancy [{query_info['language']}] {type(e).__name__}: {e}")

    run_a_scores.append(row)

run_a_scores

[{'language': 'en',
  'query': 'How much overtime pay am I entitled to?',
  'answer': 'To calculate your overtime pay in Singapore:  \n\n1. **Eligibility**:  \n   - **Non-workmen**: Monthly basic salary ≤ $2,600.  \n   - **Workmen**: Monthly basic salary ≤ $4,500.  \n\n2. **Overtime Rate**:  \n   - Minimum **1.5 times** your hourly basic rate.  \n   - For **non-workmen**, the hourly rate is capped at **$13.60** (so overtime pay = $13.60 × 1.5 × hours).  \n\n3. **Hourly Basic Rate Calculation**:  \n   - **Monthly-rated**: (12 × monthly salary) ÷ (52 × 44).  \n   - **Daily-rated**: Daily pay ÷ working hours per day.  \n   - **Piece-rated**: Weekly pay ÷ total hours worked in the week.  \n\n4. **Maximum Overtime per Month**:  \n   - **72 hours** (exceeding this requires an exemption).  \n\n5. **Payment Timing**:  \n   - Overtime pay must be paid **within 14 days** of the salary period.  \n\nIf you work on a **rest day or public holiday**, additional rules apply (e.g., rest day pay + overt

In [10]:
print("=== Run A, trimmed (dense retrieval, qwen3:8b, 8 queries) ===\n")

for row in run_a_scores:
    faithfulness = row["faithfulness"]
    relevancy = row["answer_relevancy"]
    flag = ""
    if faithfulness is None or faithfulness < 0.7:
        flag = "  <-- review"
    if relevancy is None or relevancy < 0.5:
        flag = "  <-- review"
    print(
        f"[{row['language']}] faithfulness={faithfulness}  answer_relevancy={relevancy}{flag}  "
        f"{row['query'][:50]}"
    )
    if flag:
        print(f"    answer: {row['answer']}")

run_a_valid_faithfulness = [r["faithfulness"] for r in run_a_scores if r["faithfulness"] is not None]
run_a_valid_relevancy = [r["answer_relevancy"] for r in run_a_scores if r["answer_relevancy"] is not None]

run_a_avg_faithfulness = (
    sum(run_a_valid_faithfulness) / len(run_a_valid_faithfulness) if run_a_valid_faithfulness else None
)
run_a_avg_relevancy = (
    sum(run_a_valid_relevancy) / len(run_a_valid_relevancy) if run_a_valid_relevancy else None
)

print(f"\nAvg faithfulness ({len(run_a_valid_faithfulness)}/{len(run_a_scores)} scored): "
      f"{run_a_avg_faithfulness:.3f}" if run_a_avg_faithfulness is not None else "\nNo faithfulness scores succeeded")
print(f"Avg answer_relevancy ({len(run_a_valid_relevancy)}/{len(run_a_scores)} scored): "
      f"{run_a_avg_relevancy:.3f}" if run_a_avg_relevancy is not None else "No answer_relevancy scores succeeded")

=== Run A, trimmed (dense retrieval, qwen3:8b, 8 queries) ===

[en] faithfulness=0.8333333333333334  answer_relevancy=0.734755041609538  How much overtime pay am I entitled to?
[en] faithfulness=1.0  answer_relevancy=0.8106790156404573  When must my employer pay my salary?
[ms] faithfulness=0.8333333333333334  answer_relevancy=0.8780610703712775  Bilakah majikan saya perlu bayar gaji saya?
[ta] faithfulness=0.8461538461538461  answer_relevancy=0.7397435278716212  எனக்கு எவ்வளவு கூடுதல் நேர ஊதியம் கிடைக்கும்?
[vi] faithfulness=1.0  answer_relevancy=0.823090998310647  Chủ sử dụng lao động của tôi phải trả lương khi nà
[en] faithfulness=1.0  answer_relevancy=0.859860240920938  Who pays repatriation costs when my Work Permit en
[en] faithfulness=0.75  answer_relevancy=0.7668510141964369  How much medical insurance must my employer provid
[en] faithfulness=0.09090909090909091  answer_relevancy=0.9938365317736336  <-- review  How can I contact MOM?
    answer: To contact MOM:  
1. **Visit lo

In [11]:
import datetime

RESULTS_DIR = Path("../data/processed/eval_results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

run_a_output = {
    "notebook": "06_eval_step7_run_a_trimmed",
    "run": "A_trimmed_8q",
    "retrieval_method": "dense",
    "generation_model": OLLAMA_MODEL_NAME,
    "judge_model": JUDGE_MODEL_NAME,
    "generated_at": datetime.datetime.now().isoformat(),
    "results": [
        {**row, "answer_relevancy": float(row["answer_relevancy"]) if row["answer_relevancy"] is not None else None}
        for row in run_a_scores
    ],
    "avg_faithfulness": run_a_avg_faithfulness,
    "avg_answer_relevancy": run_a_avg_relevancy,
}

run_a_output_path = RESULTS_DIR / "06_eval_run_a_trimmed_8q.json"
run_a_output_path.write_text(json.dumps(run_a_output, indent=2, ensure_ascii=False), encoding="utf-8")
run_a_output_path

WindowsPath('../data/processed/eval_results/06_eval_run_a_trimmed_8q.json')